

```
https://drive.google.com/file/d/1Lt-H7ul_t6DUgBg3fJnRir1_c0_RF4iZ/view?usp=sharing
```



In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
import kagglehub
adhamashraf202200953_technical_parsed_questions_path = kagglehub.dataset_download('adhamashraf202200953/technical-parsed-questions')

Using Colab cache for faster access to the 'technical-parsed-questions' dataset.


In [ ]:
# Upgrade PyTorch to a unified CUDA 12.1 version
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install and upgrade all Hugging Face and evaluation libraries together
!pip install --upgrade transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score bert_score nltk

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
# !pip install --upgrade torchao

In [ ]:
def migrate_essay_data(folder_path):
    all_essay_records = []
    search_pattern = os.path.join(folder_path, "*_essay.jsonl")
    essay_files = glob.glob(search_pattern)
    for file_path in essay_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        record = json.loads(line)
                        if record.get("type") == "essay" and record.get("model_answer"):
                            all_essay_records.append(record)
                    except: continue
    return all_essay_records

def preprocess_function(examples):
    full_texts = []
    for c, s, q, a in zip(examples["context"], examples["scenario_context"], examples["question"], examples["model_answer"]):
        text = f"Context: {c}\n\nScenario: {s} Question: {q} Answer: {a}{tokenizer.eos_token}"
        full_texts.append(text)

    model_inputs = tokenizer(full_texts, max_length=MAX_LENGTH, truncation=True)
    return model_inputs


In [ ]:
import gc
import torch
import os
import json
import glob
import numpy as np
from datasets import Dataset
from transformers import (
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-1B"
DATA_FOLDER = "/kaggle/input/technical-parsed-questions"
OUTPUT_DIR = "./outputs/STABLE_LLAMA"

BATCH_SIZE = 2
GRADIENT_ACC_STEPS = 16
MAX_LENGTH = 512

gc.collect()
torch.cuda.empty_cache()

In [ ]:
raw_essay_data = migrate_essay_data(DATA_FOLDER)
dataset = Dataset.from_list(raw_essay_data)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- CONFIGURING PADDING FOR LLAMA ---
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Standard formatting for causal training

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)

print("Total training samples:", len(split_dataset["train"]))

Map:   0%|          | 0/2209 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Total training samples: 1988


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=5,
    learning_rate=2e-4,
    num_train_epochs=3,
    max_grad_norm=1.0,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

Step,Training Loss,Validation Loss
50,1.722042,1.702306
100,1.628540,1.656572
150,1.543892,1.640956
189,1.561124,1.634833


TrainOutput(global_step=189, training_loss=1.676219861974161, metrics={'train_runtime': 1336.1336, 'train_samples_per_second': 4.464, 'train_steps_per_second': 0.141, 'total_flos': 1.5646356336795648e+16, 'train_loss': 1.676219861974161, 'epoch': 3.0})

In [ ]:
# --- UPDATED OUTPUT COMPRESSION ---
!zip -r "/content/LLAMA-3.2-1B.zip" "/content/outputs"

# ==========================================
# INFERENCE BLOCK WITH FINETUNED ADAPTERS
# ==========================================
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "meta-llama/Llama-3.2-1B"
LLAMA_LORA_DIR = "/content/outputs/STABLE_LLAMA/checkpoint-189" # Double-check your final step number!

print("Loading Llama 3.2...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, LLAMA_LORA_DIR)

def ask_llama(context_text):
    prompt = f"Context: {context_text}\n\nScenario:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    generated_answer = full_text[len(prompt):].strip()
    return f"Scenario: {generated_answer}"

sample_context = """
When a Docker container exits immediately with a status code 0, it typically means the primary
foreground process specified in the ENTRYPOINT or CMD has completed its execution. Containers
require a persistent foreground process to remain active. To keep a debugging container running
without an active service, developers often append a blocking command like 'tail -f /dev/null'
or run the container interactively using the '-it' flags, though this can consume unnecessary
idle system resources.
"""
print(ask_llama(sample_context))

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/STABLE_LLAMA/ (stored 0%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/ (stored 0%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/training_args.bin (deflated 54%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/scaler.pt (deflated 64%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/rng_state.pth (deflated 26%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/adapter_config.json (deflated 60%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/trainer_state.json (deflated 75%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/scheduler.pt (deflated 62%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/README.md (deflated 65%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/tokenizer_config.json (deflated 45%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/adapter_model.safetensors (deflated 8%)
  adding: content/outputs/STABLE_LLAMA/checkpoint-189/tokenizer.json (

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Scenario: You are tasked with developing a monitoring system for your company's production containers. The system must continuously track the health of each container and take appropriate action if necessary. However, you've noticed that some containers are exiting immediately with status code 0, leading to unreliable monitoring data. Question: Explain how you would design your monitoring system to handle containers that exit immediately with status code 0. Be sure to address the tradeoffs between continuous monitoring and resource consumption. Answer: Implement a periodic check on each container's status, periodically checking for events like container creation or termination. If a container exits with status code 0, take appropriate action, such as restarting the container or logging the event. This approach balances continuous monitoring with resource consumption by avoiding unnecessary idle system resources.
